#Introduction

This notebook demonstrates the difference between Similarity Search and Maximum Marginal Relevance (MMR) in a Retrieval-Augmented Generation (RAG) system.

In RAG, retrieval quality directly affects the quality of answers produced by an LLM.
Two common retrieval strategies are:

* Similarity Search – retrieves the most relevant documents

* MMR (Maximum Marginal Relevance) – retrieves relevant and diverse documents

This notebook shows how both behave differently on the same data.
#What this notebook contains

This notebook includes:

* A small set of intentionally redundant documents

* A vector database built using embeddings

* A Similarity retriever

* An MMR retriever

* A side-by-side comparison of retrieved results

* Experiments with MMR lambda values

#Step 1: Install required libraries

In [ ]:
!pip install sentence-transformers numpy -q

#Step 2: Import required classes

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

#Step 3: Create sample documents

In [ ]:
chunks = [
    "Employees are entitled to 20 days of paid leave per year.",
    "The leave policy includes sick leave and casual leave.",
    "Employees must submit leave requests through the HR portal.",
    "The office cafeteria opens at 9 AM and closes at 6 PM.",
    "Parking is available for employees in the basement."
]

#Step 4: Load the embedding model

In [ ]:
chunk_embeddings = model.encode(chunks)
chunk_embeddings.shape

#Step 5: Define the user query

In [ ]:
query = "What is the employee leave policy?"
query_embedding = model.encode(query)

#Step 6: Similarity Retrieval (Top-k)

In [ ]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


In [ ]:
similarity_scores = []

for i, emb in enumerate(chunk_embeddings):
    score = cosine_similarity(query_embedding, emb)
    similarity_scores.append((chunks[i], score))

similarity_scores


In [ ]:
top_k = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

for text, score in top_k:
    print(f"Score: {score:.4f} | Chunk: {text}")

#Step 7: MMR Retrieval
What changes here?

Only the retrieval strategy.

MMR:

* Starts with relevant documents

* Penalizes redundancy

* Selects documents that add new information

In [ ]:
def mmr(query_embedding, chunk_embeddings, chunks, top_k=3, lambda_param=0.7):
    selected = []
    selected_indices = []

    similarities = [
        cosine_similarity(query_embedding, emb)
        for emb in chunk_embeddings
    ]

    first = np.argmax(similarities)
    selected.append(chunks[first])
    selected_indices.append(first)

    for _ in range(top_k - 1):
        mmr_scores = []
        for i in range(len(chunks)):
            if i in selected_indices:
                continue
            relevance = similarities[i]
            diversity = max(
                cosine_similarity(chunk_embeddings[i], chunk_embeddings[j])
                for j in selected_indices
            )
            score = lambda_param * relevance - (1 - lambda_param) * diversity
            mmr_scores.append((i, score))

        next_idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected.append(chunks[next_idx])
        selected_indices.append(next_idx)

    return selected

In [ ]:
mmr_results = mmr(query_embedding, chunk_embeddings, chunks, top_k=3)

for chunk in mmr_results:
    print(chunk)

#Final takeaway

* Similarity search maximizes relevance

* MMR balances relevance and diversity

* Both use the same vector database

* Only the selection logic changes

In RAG systems, MMR often produces better grounding context for LLMs, especially for explanatory questions.